# Qwen3 1.7B Core ML quantization

✔ PyTorch 2.6
✔ transformers 4.51
✔ coremltools 8.0
✔ torchvision 0.21.0
✔ torchaudio 2.6.0

In [1]:
import torch
import coremltools as ct
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-1.7B" 
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, attn_implementation="eager", low_cpu_mem_usage=True)

scikit-learn version 1.7.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.6.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.4.0 is the most recent version that has been tested.
/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.86s/it]


In [2]:
model = model.to(torch.float16)
model.to("cpu")
model.config.use_cache = False
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwe

In [ ]:
for name, b in model.named_buffers():
    if b.dtype != torch.float16:
        print("BUFFER NOT FP16:", name, b.dtype)

# check the dtypes of the model parameters - {torch.float16}
unique_dtypes = set(p.dtype for p in model.parameters())
print(unique_dtypes)

# check rotary embeddings
#model.model.rotary_emb.inv_freq = model.model.rotary_emb.inv_freq.to(torch.float16)
print(model.model.rotary_emb.inv_freq.dtype)

In [ ]:
# dry run

# torch.Size([1, 1, 151936])

input_ids = torch.zeros((1, 1), dtype=torch.long)
with torch.no_grad():
    outputs = model(input_ids)

print(outputs.logits.shape)

In [ ]:
example_input = torch.zeros((1, 2048), dtype=torch.long)
#model.config._attn_implementation = "eager"


for p in model.parameters():
    assert p.dtype == torch.float16

print(model.config._attn_implementation)

for name, module in model.named_modules():
    if "attn" in name.lower():
        print(name, type(module))


## Convert to CoreML - Exported program

In [3]:
# torch export

#MAX_SEQ_LEN = 2048
#MAX_SEQ_LEN = 1024
MAX_SEQ_LEN = 512
#DEFAULT_SEQ_LEN = 128
DEFAULT_SEQ_LEN = 32

# context length
example_input = (torch.zeros((1, DEFAULT_SEQ_LEN), dtype=torch.long),) # Flexible input shape
#example_input = (torch.zeros((1, 1024), dtype=torch.long),)

from torch.export import Dim

seq_dim = Dim("seq_len", min=1, max=MAX_SEQ_LEN) # Flexible input shape


from torch.export import export

aten_ops = list(torch.ops.aten.__dict__.values())

exported_program = export(
    model,
    example_input,
    dynamic_shapes={ # Trying to optimize loading
        "input_ids": {1: torch.export.Dim("seq_len", min=1, max=MAX_SEQ_LEN)}
    },
)

# Run decompositions
exported_program = exported_program.run_decompositions({})


In [ ]:
print(exported_program.dialect)
print(type(exported_program))

graph_str = str(exported_program.graph_module.graph)
print(exported_program.dialect)
print("alias present:", "alias" in str(exported_program.graph_module.graph))
print("aten.diff present:", "aten.diff" in graph_str)

In [ ]:
from collections import defaultdict

op_summary = defaultdict(int)
unsupported = []

for node in exported_program.graph_module.graph.nodes:
    target = node.target
    
    # Only inspect callable ops
    if hasattr(target, "__module__"):
        module_name = target.__module__
        op_summary[module_name] += 1
        
        # Flag non-aten namespaces
        if not module_name.startswith("torch._ops.aten"):
            unsupported.append((module_name, target))
    else:
        # Non-callable targets (rare, but include for completeness)
        op_summary[str(type(target))] += 1

print("\n=== Operator Modules Present ===")
for k, v in op_summary.items():
    print(f"{k} -> {v}")

print("\n=== Unsupported Ops Detected ===")
for mod, op in unsupported:
    print(f"{mod} :: {op}")


In [ ]:
# Convert to Core ML FP16 - Direct

import coremltools as ct
import numpy as np

mlmodel = ct.convert(
    exported_program,
    convert_to="mlprogram",
    compute_precision=ct.precision.FLOAT16,
    #compute_units=ct.ComputeUnit.CPU_AND_GPU,
    minimum_deployment_target=ct.target.iOS18,
    skip_model_load=True,
    #package_dir="models/Qwen3_1_7B_fp16_2048_flex_def_128_cpu_gpu.mlpackage",
    #package_dir="models/Qwen3_1_7B_fp16_1024_flex_def_128.mlpackage",
    package_dir="models/Qwen3_1_7B_fp16_512_flex_def_32.mlpackage",
    debug=True,
    #states=states,    
    #pass_pipeline=ct.PassPipeline.DEFAULT,
    #state_implementation="MLState"
)

Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 30.76 passes/s]


: 

In [ ]:
import coremltools as ct
#mlmodel = ct.models.MLModel("models/Qwen3_1_7B_fp16_2048_flex_def_128.mlpackage") # compute unit = ALL
#mlmodel = ct.models.MLModel("models/Qwen3_1_7B_fp16_2048_flex_def_128_cpu_gpu.mlpackage") # compute unit = CPU_AND_GPU - crashes the kernel
#mlmodel = ct.models.MLModel("models/Qwen3_1_7B_fp16_2048_flex_def_128.mlpackage", compute_units=ct.ComputeUnit.CPU_AND_GPU) # compute unit = CPU_AND_GPU - crashes the kernel
mlmodel = ct.models.MLModel("models/Qwen3_1_7B_fp16_2048_flex_def_128.mlpackage", compute_units=ct.ComputeUnit.CPU_ONLY)

In [ ]:
spec = mlmodel.get_spec()

for inp in spec.description.input:
    print("Input name:", inp.name)
    
    if inp.type.WhichOneof("Type") == "multiArrayType":
        ma = inp.type.multiArrayType
        
        # If flexible shape
        if ma.shapeRange.sizeRanges:
            print("  Flexible shape ranges:")
            for i, r in enumerate(ma.shapeRange.sizeRanges):
                print(f"    Dim {i}: min={r.lowerBound}, max={r.upperBound}")
            
            # Default shape (if explicitly stored)
            if len(ma.shape) > 0:
                print("  Default shape:", list(ma.shape))
            else:
                print("  Default shape not explicitly set (uses lowerBound)")
        
        # If fixed shape
        elif len(ma.shape) > 0:
            print("  Fixed shape:", list(ma.shape))

## Weight 4-bit quantization

In [ ]:
from coremltools.optimize.coreml import linear_quantize_weights
import coremltools.optimize as cto


config = cto.coreml.OptimizationConfig(
        global_config=cto.coreml.OpLinearQuantizerConfig(mode="linear_symmetric", dtype="int4")
    )

quantized_model = linear_quantize_weights(
    mlmodel,
    config=config,
    joint_compression=True
)

#quantized_model.save("models/Qwen3_1_7B_w4.mlpackage")
#quantized_model.save("models/Qwen3_1_7B_w4_2048_flex_def_128_cpu_gpu.mlpackage")
quantized_model.save("models/Qwen3_1_7B_w4_1024_flex_def_128.mlpackage")

In [ ]:
## Loading 4-bit quantized model and checking if it is exapnded to fp16 during runtime

mlmodel = ct.models.MLModel("models/Qwen3_1_7B_w4_2048_flex_def_128_cpu_gpu.mlpackage", compute_units=ct.ComputeUnit.CPU_ONLY)

In [ ]:
import coremltools as ct
from google.protobuf import text_format

spec = mlmodel.get_spec()

print("========== BASIC INFO ==========")
print("Spec version:", spec.specificationVersion)
print("Model type:", spec.WhichOneof("Type"))

print("\n========== INPUT INFO ==========")
for inp in spec.description.input:
    print("Input:", inp.name)
    if inp.type.WhichOneof("Type") == "multiArrayType":
        ma = inp.type.multiArrayType
        
        if len(ma.shape) > 0:
            print("  Default shape:", list(ma.shape))
        
        if ma.shapeRange.sizeRanges:
            print("  Flexible ranges:")
            for i, r in enumerate(ma.shapeRange.sizeRanges):
                print(f"    Dim {i}: min={r.lowerBound}, max={r.upperBound}")

print("\n========== TEXT SEARCH ==========")

spec_text = text_format.MessageToString(spec)

keywords = [
    "quantize",
    "dequantize",
    "constexpr_lut_to_dense",
    "constexpr",
    "int4",
    "int8",
    "blockwise",
    "linear_quantize"
]

for kw in keywords:
    print(f"{kw}: {kw in spec_text}")

print("\n========== OP COUNTS ==========")

# Simple heuristic counts
print("quantize ops count:", spec_text.count("quantize"))
print("dequantize ops count:", spec_text.count("dequantize"))
print("constexpr ops count:", spec_text.count("constexpr"))

## Weight 8 bit quantization

May be more runtime memory efficient if even if it has larger disk size

In [ ]:
mlmodel = ct.models.MLModel("models/Qwen3_1_7B_fp16_2048_flex_def_128.mlpackage", compute_units=ct.ComputeUnit.CPU_ONLY)

In [ ]:
from coremltools.optimize.coreml import linear_quantize_weights
import coremltools.optimize as cto


config = cto.coreml.OptimizationConfig(
        global_config=cto.coreml.OpLinearQuantizerConfig(mode="linear", dtype="int8")
    )

quantized_model = linear_quantize_weights(
    mlmodel,
    config=config,
    joint_compression=True
)

#quantized_model.save("models/Qwen3_1_7B_w8_2048_flex_def_128.mlpackage")
#quantized_model.save("models/Qwen3_1_7B_w8_1024_flex_def_128.mlpackage")
quantized_model.save("models/Qwen3_1_7B_w8_512_flex_def_32.mlpackage")

Running compression pass linear_quantize_weights: 100%|██████████| 315/315 [00:41<00:00,  7.67 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running MIL default pipeline:  36%|███▌      | 31/86 [00:19<00:01, 27.94 passes/s]

## Weight + activation calibration

In [ ]:
# Numeric calibration

import numpy as np

calibration_samples = []

for _ in range(200):
    #seq_len = np.random.randint(1, 2048) #Takes too much memory 
    seq_len = np.random.choice([32, 64, 128, 256, 512])
    tokens = np.random.randint(
        low=0,
        high=151936,  # vocab size / to be changed based on the actual tokenizer vocab size
        size=(1, seq_len),
        dtype=np.int32
    )
    calibration_samples.append({"input_ids": tokens})

print(calibration_samples[0])

In [ ]:
#  Text calibration block: realistic empathy / mental-health prompts

import random
import numpy as np
from transformers import AutoTokenizer

TOKENIZER_NAME = "Qwen/Qwen3-1.7B"
NUM_CALIBRATION_SAMPLES = 64   # 128–200 is ideal
#LENGTH_CHOICES = [32, 64, 128, 256, 512]
LENGTH_CHOICES = [64]

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=False)

# --- Components for synthetic empathy prompts ---

feelings = [
    "anxious", "overwhelmed", "lonely", "sad", "burned out",
    "numb", "restless", "hopeless", "stressed", "ashamed",
    "guilty", "confused", "unmotivated", "exhausted"
]

situations = [
    "at work", "in my relationship", "about my future",
    "with my family", "about my health", "about money",
    "in social situations", "during meetings",
    "before bed", "when I wake up"
]

requests = [
    "What can I do to cope?",
    "How can I manage this feeling?",
    "What small steps can I take?",
    "Can you give me grounding exercises?",
    "How can I be kinder to myself?",
    "What strategies might help?",
    "How do I explain this to someone close to me?",
    "How can I build resilience?",
    "How do I calm down in the moment?",
    "What habits can improve this over time?"
]

supportive_frames = [
    "I’ve been feeling {feeling} {situation}. {request}",
    "Lately I feel {feeling}, especially {situation}. {request}",
    "I’m struggling with feeling {feeling} {situation}. {request}",
    "Sometimes I feel very {feeling} {situation}. {request}",
    "I notice I become {feeling} {situation}. {request}",
]

calibration_samples = []

for _ in range(NUM_CALIBRATION_SAMPLES):

    template = random.choice(supportive_frames)

    text = template.format(
        feeling=random.choice(feelings),
        situation=random.choice(situations),
        request=random.choice(requests),
    )

    # Occasionally extend text for more realistic distribution
    if random.random() < 0.25:
        text += " I want practical advice that feels manageable and realistic."
    if random.random() < 0.20:
        text += " I would appreciate a calm and understanding tone."

    seq_len = int(np.random.choice(LENGTH_CHOICES))

    tok = tokenizer(
        text,
        max_length=seq_len,
        #max_length=256,
        truncation=True,
        padding="max_length",
        return_tensors="np"
    )

    input_ids = tok["input_ids"].astype(np.int32)

    # Add tiny random token perturbation
    noise_mask = np.random.rand(*input_ids.shape) < 0.01
    random_tokens = np.random.randint(0, tokenizer.vocab_size, size=input_ids.shape)
    input_ids = np.where(noise_mask, random_tokens, input_ids)
    input_ids = input_ids.astype(np.int32)

    calibration_samples.append({"input_ids": input_ids})

print(f"Built {len(calibration_samples)} calibration samples.")
print("Example shape:", calibration_samples[0]["input_ids"].shape)
#print(calibration_samples[0])
mlmodel.predict(calibration_samples[0])


In [ ]:
# Check for zero tensors using Pytorch model

import torch

EPS = 1e-8
problem_layers = set()

# Make sure you already created tokenizer
# tokenizer = AutoTokenizer.from_pretrained("your-qwen-model")

def check_stats(name):
    def hook(module, inputs, output):

        # Handle tuple outputs
        if isinstance(output, tuple):
            outputs = [o for o in output if isinstance(o, torch.Tensor)]
        elif isinstance(output, torch.Tensor):
            outputs = [output]
        else:
            return

        for out in outputs:
            if out.numel() == 0:
                continue

            max_abs = torch.max(torch.abs(out)).item()
            min_val = torch.min(out).item()
            max_val = torch.max(out).item()
            range_val = max_val - min_val

            if max_abs == 0:
                print(f"[ZERO] {name}")
                problem_layers.add(name)

            elif range_val < EPS:
                print(f"[LOW RANGE] {name} | range={range_val:.3e}")
                problem_layers.add(name)

    return hook


# Attach hooks to important modules
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        module.register_forward_hook(check_stats(name))

    if isinstance(module, torch.nn.LayerNorm):
        module.register_forward_hook(check_stats(name))


# Run calibration samples (TEXT → TOKENIZED)
model.eval()
with torch.no_grad():

    for sample in calibration_samples:

        input_ids = torch.from_numpy(sample["input_ids"])

        # Move to same device as model
        #input_ids = input_ids.to(next(model.parameters()).device)

        model(input_ids)


print("\nLayers with zero/low-range activations:")
for layer in problem_layers:
    print(layer)

In [ ]:
# Check for constant tensors leading to zero activations

import torch

EPS = 1e-7
constant_layers = []

def check_constant(name):
    def hook(module, inputs, output):

        if isinstance(output, tuple):
            outputs = [o for o in output if isinstance(o, torch.Tensor)]
        elif isinstance(output, torch.Tensor):
            outputs = [output]
        else:
            return

        for out in outputs:
            if out.numel() == 0:
                continue

            min_val = out.min().item()
            max_val = out.max().item()
            range_val = max_val - min_val

            if range_val < EPS:
                print(f"[CONSTANT] {name} | range={range_val:.3e}")
                constant_layers.append(name)

    return hook


# Attach to ALL modules
for name, module in model.named_modules():
    module.register_forward_hook(check_constant(name))


# Run calibration
model.eval()
with torch.no_grad():
    for sample in calibration_samples:

        sample = sample["input_ids"]

        if not isinstance(sample, torch.Tensor):
            input_ids = torch.from_numpy(sample)
        else:
            input_ids = sample

        input_ids = input_ids.long()
        if input_ids.dim() == 1:
            input_ids = input_ids.unsqueeze(0)

        input_ids = input_ids.to(next(model.parameters()).device)

        model(input_ids)


print("\nConstant-output layers:")
for layer in set(constant_layers):
    print(layer)

In [ ]:
# Test all calibration samples on the Core ML model

for i, sample in enumerate(calibration_samples):
    try:
        mlmodel.predict(sample)
        print(i)
    except Exception as e:
        print("Failed at sample", i)
        print(sample["input_ids"].shape)
        raise e

In [ ]:
from coremltools.optimize import coreml as cto_coreml

#activation_config = cto_coreml.OptimizationConfig(
#    global_config=cto_coreml.experimental.OpActivationLinearQuantizerConfig(
#        mode="linear_symmetric"
#    )
#)

activation_config = cto_coreml.OptimizationConfig(
    global_config=None,  # Do NOT quantize everything
    op_type_configs={
        "linear": cto_coreml.experimental.OpActivationLinearQuantizerConfig(
            mode="linear_symmetric"  # Use 'linear' to avoid 0 scale warning? - linear is not available for activations in the current version of coremltools, so using linear_symmetric instead
        ),
        "matmul": cto_coreml.experimental.OpActivationLinearQuantizerConfig(
            mode="linear_symmetric"
        ),
        #"add": cto_coreml.experimental.OpActivationLinearQuantizerConfig(
        #    mode="linear_symmetric"
        #),
    }
)

a8_model = cto_coreml.experimental.linear_quantize_activations(
    mlmodel,
    activation_config,
    calibration_samples,
)

#a8_model.save("models/Qwen3_1_7B_A8_2048_flex_def_128.mlpackage")
a8_model.save("models/Qwen3_1_7B_A8_2048_flex_def_128_cpu.mlpackage")


#Activation quantization is FAILING even with CPU only - Possible issue difference in tolerance resolution bertween pytorch and coremltools.
#linear_quantize_activations failed with compute unit = ALL (ANE error)
#CPU AND GPU is crashing the kernel during model load itself

## Mixed precision

In [ ]:
# Mixed precision

import coremltools as ct
import numpy as np

from coremltools.optimize.coreml import (
    linear_quantize_weights,
    OptimizationConfig,
    OpLinearQuantizerConfig
)

weight_config = OptimizationConfig(
    global_config=OpLinearQuantizerConfig(
        mode="linear_symmetric",
        dtype="int4",
        granularity="per_block",
        block_size=32
    ),
    op_type_configs={
        "layer_norm": None,
        #"embedding": None,
        "softmax": None,
        "rms_norm": None  # if present
    }
)

In [ ]:
a8_model = ct.models.MLModel("models/Qwen3_1_7B_A8_2048_flex_def_128_cpu.mlpackage", compute_units=ct.ComputeUnit.CPU_ONLY)

w4a8_model = linear_quantize_weights(
    a8_model,
    config=weight_config,
    joint_compression=True 
)

#w4a8_model.save("models/Qwen3_1_7B_W4A8_emb_fp16_2048_flex_def_128_cpu.mlpackage")
w4a8_model.save("models/Qwen3_1_7B_W4A8_emb_w4_2048_flex_def_128_cpu.mlpackage")

In [ ]:
#Check if weights will be expanded to fp16

spec = a8_model.get_spec()
txt = str(spec)
print("constexpr_lut_to_dense" in txt)


In [ ]:
# Test inference

test_input = {
    "input_ids": np.random.randint(
        0,
        151936,
        size=(1, 128),
        dtype=np.int32
    )
}

output = w4a8_model.predict(test_input)
print("Inference OK")

# Misc code

## Export through Trace - Flex context length

In [ ]:
# trace 
import torch._logging
import logging
torch._logging.set_logs(dynamo=logging.DEBUG, graph_code=True)

with torch.no_grad():
    traced_model = torch.jit.trace(
        model,
        example_input,
        check_trace=False,
        strict=False
    )

print(traced_model.graph)

# scripted model
#scripted_model = torch.jit.script(model)

In [ ]:
print("CHECKS")
graph_str = str(traced_model.graph)

print("---- Checking for FP32 in graph ----")

has_fp32 = False

for node in traced_model.graph.nodes():
    for output in node.outputs():
        if "Float" in str(output.type()):
            print("FP32 node:", node)
            has_fp32 = True

if not has_fp32:
    print("✅ Graph is clean FP16")
print("------------------")

checks = {
    "aten::diff present": "aten::diff" in graph_str,
    "prims present": "prims::" in graph_str,
    "alias present": "alias" in graph_str,
    "higher_order present": "higher_order" in graph_str,
    "training dialect present": "training" in graph_str,
}

for k, v in checks.items():
    print(f"{k}: {v}")

from coremltools.converters.mil.frontend.torch.ops import _TORCH_OPS_REGISTRY

graph_str = str(traced_model.graph)

used_ops = set()

for line in graph_str.split("\n"):
    line = line.strip()
    
    if " = aten::" in line:
        op = line.split(" = ")[1].split("(")[0]
        used_ops.add(op.replace("aten::", ""))

print("=== ATEN Ops Used In Model ===")
for op in sorted(used_ops):
    print(op)

print("\nTotal used ATEN ops:", len(used_ops))

supported_ops = set(_TORCH_OPS_REGISTRY.name_to_func_mapping.keys())

print("Number of supported ATEN ops:", len(supported_ops))
unsupported = []

for op in used_ops:
    op_name = op.replace("aten::", "")
    #op_name = op
    if op_name not in supported_ops:
        unsupported.append(op)

print("Unsupported ATEN ops:")
for op in unsupported:
    print(op)

In [ ]:
print(traced_model.code)

In [ ]:
# Through jit trace

import coremltools as ct
import numpy as np

input_shape = ct.EnumeratedShapes(
    shapes=[
        [1, 128],
        [1, 256],
        [1, 512],
        [1, 1024],
        [1, 2048],
    ],
    default=[1, 128],
)

mlmodel = ct.convert(
    traced_model,
    convert_to="mlprogram",
    compute_precision=ct.precision.FLOAT16,
    minimum_deployment_target=ct.target.iOS18,
    skip_model_load=True,
    package_dir="models/Qwen3_1_7B_fp16_2048_flex_def_128.mlpackage",
    inputs=[ct.TensorType(shape=input_shape, dtype=np.int32)],
    debug=True,
)


### Flex context length (NOT TESTED)

In [ ]:
import coremltools as ct
import torch

# Define dynamic dimension
seq_dim = ct.RangeDim(lower_bound=1, upper_bound=2048)

mlmodel = ct.convert(
    exported_program,
    convert_to="mlprogram",
    inputs=[
        ct.TensorType(
            name="input_ids",
            shape=(1, seq_dim),
            dtype=int,
        )
    ],
)

mlmodel.save("Qwen3_1_7B_fp16_2048_flex.mlpackage")

In [ ]:
from coremltools.optimize.coreml import linear_quantize_weights
import coremltools.optimize as cto


config = cto.coreml.OptimizationConfig(
        global_config=cto.coreml.OpLinearQuantizerConfig(mode="linear_symmetric", dtype="int4")
    )

quantized_model = linear_quantize_weights(
    mlmodel,
    config=config,
    joint_compression=True
)

quantized_model.save("models/Qwen3_1_7B_w4_2048_flex.mlpackage")

In [ ]:
import numpy as np
states = []
for layer_idx in range(28):  # num_hidden_layers
    key_state = ct.StateType(
        wrapped_type=ct.TensorType(
            shape=(1, 8, ct.RangeDim(0, 2048), 128),  # Dynamic seq_len
            dtype=np.float16
        ),
        name=f"key_cache_{layer_idx}"
    )
    value_state = ct.StateType(
        wrapped_type=ct.TensorType(
            shape=(1, 8, ct.RangeDim(0, 2048), 128),  # Dynamic seq_len
            dtype=np.float16
        ),
        name=f"value_cache_{layer_idx}"
    )
    states.append(key_state)
    states.append(value_state)

In [ ]:
for node in exported_program.graph_module.graph.nodes:
    if "diff" in str(node.target):
        print("Node:", node)
        print("Inputs:", node.args)

for node in exported_program.graph_module.graph.nodes:
    if "diff" in str(node.target):
        print("Node:", node)
        print("Target:", node.target)
        print("Args:", node.args)
        print("Users:", list(node.users))
        print("-" * 50)

for node in exported_program.graph_module.graph.nodes:
    if "diff" in str(node.target):
        diff_node = node
        break

print("Diff node:", diff_node)
print("Input node:", diff_node.args[0])
producer = diff_node.args[0]
print("Producer:", producer)
print("Producer target:", producer.target)
print("Producer args:", producer.args)

In [ ]:
import transformers.models.qwen3.modeling_qwen3 as modeling_qwen3
import inspect

print(inspect.getsourcefile(modeling_qwen3))

source_lines, start_line = inspect.getsourcelines(modeling_qwen3)

for i, line in enumerate(source_lines):
    if "arange" in line:
        l_no = i
        print(f"{start_line + i}: {line.strip()}")

source_lines, start_line = inspect.getsourcelines(modeling_qwen3)

for j in range(l_no-8, l_no+12):
    print(f"{start_line + j}: {source_lines[j].rstrip()}")

print("---------------------")

for j in range(390, 480):
    print(f"{start_line + j}: {source_lines[j].rstrip()}")


In [ ]:
for i, line in enumerate(source_lines):
    if "def create_causal_mask" in line:
        print(f"{start_line + i}: {line.strip()}")

print("---------------------")

for line in source_lines:
    stripped = line.strip()
    
    # Stop when first class or function appears
    if stripped.startswith("class ") or stripped.startswith("def "):
        break
    
    print(line.rstrip())

In [ ]:
#import transformers.masking_utils as masking_utils
import inspect

#print(inspect.getsourcefile(masking_utils))

#source_lines, start_line = inspect.getsourcelines(masking_utils)

for i, line in enumerate(source_lines):
    if "def create_causal_mask" in line:
        print(f"{start_line + i}: {line.strip()}")

for i, line in enumerate(source_lines):
    if "def create_causal_mask" in line:
        start_idx = i
        break

for j in range(start_idx, start_idx + 200):
    print(f"{start_line + j}: {source_lines[j].rstrip()}")


print("---------------------")

for i, line in enumerate(source_lines):
    if ".diff(" in line or "torch.diff" in line:
        print(f"{start_line + i}: {line.strip()}")

In [ ]:
#In transformers -> masking_utils.py

"""
position_diff = torch.cat(
    (
        first_dummy_value,
        position_ids[..., 1:] - position_ids[..., :-1],
    ),
    dim=-1,
)
"""